# Feature Engineering — Practice Notebook

### What is Feature Engineering?

**Feature engineering** is the process of using domain knowledge and data analysis to create, transform, or select the input variables ("features") that a machine learning model uses to learn. Raw data is rarely in a form a model can use effectively — it may have missing values, features on very different scales, categorical text labels, or it may be missing information that only becomes visible once you *combine* columns in a clever way.

### Why does it matter?

- **Garbage in, garbage out** — even the most powerful algorithm cannot compensate for poorly prepared data.
- Well-engineered features often improve model performance **more** than switching to a fancier algorithm.
- It forces you to understand your data deeply, which helps you catch errors, leakage, and bias early.
- In industry, most of a data scientist's time (often 60–80%) is spent on data preparation and feature engineering, not model building.

### What you will practice in this notebook

We will work through the full feature engineering workflow on the **Titanic** dataset (built into Seaborn):

1. Explore the raw dataset
2. Handle missing values
3. Create new features from existing ones
4. Encode categorical features
5. Scale numeric features
6. Check feature usefulness / correlation with the target

### How to use this notebook

- Read each markdown instruction cell carefully.
- Below each instruction is a code cell with `# TODO` comments. Write your own code there.
- Some cells include a `raise NotImplementedError` placeholder — **delete that line** once you've written your solution.
- Run every cell in order (Shift + Enter). Do not skip cells.
- There are **Reflection Questions** at the end of several sections — answer them in a markdown cell in your own words. Understanding *why* you did something is as important as doing it.

---


## 0. Setup

Run the cell below to import the libraries we'll need and load the Titanic dataset directly from Seaborn.

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder

pd.set_option('display.max_columns', None)

# Load the Titanic dataset built into Seaborn
df = sns.load_dataset('titanic')
df.head()


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## 1. Explore the Raw Dataset

Before you engineer anything, you must understand what you're working with.

**Your task:**
1. Print the shape of the dataset (rows, columns).
2. Print `.info()` to see data types and non-null counts.
3. Print `.describe()` for the numeric columns.
4. Print the number of missing values **per column**, sorted from most to least missing.

**Why this matters:** You cannot fix what you haven't measured. Checking shape, dtypes, and missingness up front tells you where the real feature engineering work needs to happen.

In [2]:
df.shape

(891, 15)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


In [4]:
df.describe()

,survived,pclass,age,sibsp,parch,fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [5]:
df.isnull().sum().sort_values(ascending=False)

deck           688
age            177
embarked         2
embark_town      2
sex              0
pclass           0
survived         0
fare             0
parch            0
sibsp            0
class            0
adult_male       0
who              0
alive            0
alone            0
dtype: int64

In [6]:
df.to_csv("titanic_cleaned.csv", index=False)

**Reflection Question 1:** Which columns have the most missing values? Do you think those columns are still useful, or should they be dropped? Write 2–3 sentences in the markdown cell below.

_Your answer here_

## 2. Handle Missing Values

Looking at the dataset, you should notice missing values in `age`, `embarked`, `embark_town`, and a *lot* of missing values in `deck`.

**Your task:**
1. For the `age` column: fill missing values with the **median** age (median is more robust to outliers than the mean).
2. For the `embarked` and `embark_town` columns: fill missing values with the **mode** (most frequent value) — these are categorical, so mean/median don't apply.
3. For the `deck` column: over 75% of values are missing, so imputing a single value would be misleading. Instead, create a new column `has_deck_info` that is `1` if `deck` is known and `0` if it is missing. Then you may drop the original `deck` column.
4. Confirm there are no missing values left (except any you intentionally kept) by printing missing-value counts again.

**Why this matters:** Most ML algorithms cannot handle NaN values directly. The *strategy* you choose for filling gaps (mean/median/mode vs. a "missingness indicator") should depend on why the data is missing and how much of it is missing — blindly filling everything with 0 or the mean can quietly damage your model.

In [8]:
# TODO 1: fill missing 'age' with the median


# TODO 2: fill missing 'embarked' and 'embark_town' with the mode


# TODO 3: create 'has_deck_info' (1/0) from 'deck', then drop the 'deck' column


# TODO 4: verify no unexpected missing values remain


**Reflection Question 2:** Why did we use the *median* for `age` but the *mode* for `embarked`? What would go wrong if we used the mean for a categorical column like `embarked`?

_Your answer here_

## 3. Create New Features

This is the heart of feature engineering: using domain knowledge to build new columns that expose patterns the raw columns hide.

**Your task:**
1. Create `family_size` = `sibsp` (siblings/spouses aboard) + `parch` (parents/children aboard) + 1 (the passenger themself).
2. Create `is_alone` = `1` if `family_size == 1`, else `0`.
3. Create `age_group` by binning `age` into categories: `'Child'` (0–12), `'Teen'` (13–19), `'Adult'` (20–59), `'Senior'` (60+). Hint: look up `pd.cut`.
4. Create `fare_per_person` = `fare / family_size`. (A family sharing one ticket price shouldn't each be treated as paying the full fare.)

**Why this matters:** None of these columns exist in the raw data, but each captures something meaningful — family size and being alone are known to have affected survival chances on the Titanic, and normalizing fare by family size gives a fairer sense of ticket cost per individual. This step is where you turn raw numbers into *information*.

In [ ]:
# TODO 1: create 'family_size'


# TODO 2: create 'is_alone'


# TODO 3: create 'age_group' using pd.cut with bins [0, 12, 19, 59, 100]
#         and labels ['Child', 'Teen', 'Adult', 'Senior']


# TODO 4: create 'fare_per_person'


# Preview your new columns
df[['sibsp', 'parch', 'family_size', 'is_alone', 'age', 'age_group', 'fare', 'fare_per_person']].head(10)


**Try it yourself:** Make a bar plot of survival rate by `age_group` and another by `is_alone`. Do the new features look like they carry useful signal?

In [ ]:
# TODO: bar plot of survival rate by age_group


# TODO: bar plot of survival rate by is_alone


**Reflection Question 3:** Look at your two plots. Does being alone seem to relate to survival? Does age group? Which new feature do you think will help a model more, and why?

_Your answer here_

## 4. Encode Categorical Features

Machine learning models need numbers, not text labels. We have several categorical columns: `sex`, `embarked`, `class`, `age_group`, `who`.

**Your task:**
1. Use **Label Encoding** (`LabelEncoder` or `.map()`) for `sex`, since it's a simple **binary** category (male/female).
2. Use **One-Hot Encoding** (`pd.get_dummies`) for `embarked`, `class`, and `age_group`, since these have **more than two, unordered** categories — label encoding would incorrectly imply an order between them (e.g. that 'Third' class > 'First' class numerically).
3. Combine the encoded columns back into a working dataframe called `df_encoded`.

**Why this matters:** Choosing the *wrong* encoding can actively mislead a model. Label-encoding a nominal (unordered) category makes the model think `2` is "more" than `1`, which is meaningless for something like embarkation port. One-hot encoding avoids that false ordering at the cost of extra columns.

In [ ]:
# TODO 1: label-encode 'sex' into a new column 'sex_encoded' (e.g. male=0, female=1)


# TODO 2: one-hot encode 'embarked', 'class', and 'age_group' using pd.get_dummies


# TODO 3: combine everything into df_encoded and preview it


**Reflection Question 4:** Why would it be a mistake to label-encode `embarked` (e.g. C=0, Q=1, S=2) and feed that directly into a linear model?

_Your answer here_

## 5. Scale Numeric Features

Columns like `fare` and `fare_per_person` range from 0 to over 500, while `family_size` ranges from 1 to 11. Many algorithms (e.g. KNN, SVM, logistic regression with regularization, gradient descent–based models) are sensitive to features being on very different scales.

**Your task:**
1. Pick the numeric columns that should be scaled: `age`, `fare`, `fare_per_person`, `family_size`.
2. Apply `StandardScaler` from scikit-learn to these columns (fit and transform).
3. Store the scaled values back into `df_encoded` (you can overwrite the original columns or create new `_scaled` columns — your choice, but be consistent).

**Why this matters:** Scaling doesn't change the information in a feature, only its range — but for distance-based or gradient-based algorithms, an unscaled feature with a huge range (like `fare`) can dominate the learning process purely because of its units, not because it's actually more important.

In [ ]:
# TODO 1: choose the numeric columns to scale
numeric_cols = []  # fill this in

# TODO 2: apply StandardScaler


# TODO 3: preview the scaled columns


**Reflection Question 5:** Which models actually *need* scaled features, and which ones (hint: think tree-based models) generally don't care about feature scale at all?

_Your answer here_

## 6. Check Feature Usefulness

Not every feature you create will be useful. A good feature engineer always checks their work.

**Your task:**
1. Compute the correlation of each numeric feature in `df_encoded` with the target column `survived`.
2. Sort the correlations from strongest to weakest (by absolute value).
3. Print the top 8 features by absolute correlation with `survived`.

**Why this matters:** Correlation isn't the whole story (it misses non-linear relationships), but it's a fast, simple sanity check. If a feature you engineered has almost zero correlation with the target and no clear theoretical justification, it may just be adding noise — and more noisy features can hurt a model rather than help it.

In [ ]:
# TODO 1: compute correlations with 'survived'


# TODO 2: sort by absolute value, descending


# TODO 3: print the top 8


**Reflection Question 6:** Did any of the features you engineered in Section 3 (`family_size`, `is_alone`, `fare_per_person`) end up in the top 8? What does that tell you about whether your feature engineering added value?

_Your answer here_

## Wrap-Up

You've now walked through a complete, realistic feature engineering pipeline:

1. **Explored** the raw data to find problems (missing values, mixed types).
2. **Handled missing values** with strategies matched to *why* the data was missing.
3. **Created new features** using domain reasoning, not just raw columns.
4. **Encoded categoricals** correctly based on whether categories are ordered or not.
5. **Scaled numeric features** so no column dominates purely due to its units.
6. **Validated** which features actually carry signal for the target.

**Final Challenge (optional, no code scaffold provided):** Using `df_encoded`, train a simple `LogisticRegression` or `RandomForestClassifier` to predict `survived`. Compare accuracy using (a) only the original raw numeric columns vs. (b) your fully engineered feature set. Does feature engineering improve the score?

In [ ]:
# Optional final challenge — write your own code here
